In [1]:
# !pip install langchain langchain-core langchain-community pypdf pymupdf sentence-transformers chromadb  ipywidgets 

In [2]:
from langchain_core.documents import Document

In [3]:
sample_doc = Document(
    page_content="Hello World!",
    metadata={"source": "https://www.google.com"}
)

In [4]:
sample_doc

Document(metadata={'source': 'https://www.google.com'}, page_content='Hello World!')

In [5]:
type(sample_doc)

langchain_core.documents.base.Document

In [6]:
# Text data
from langchain_community.document_loaders.text import TextLoader

loader = TextLoader("data/Python.txt", encoding="utf-8")

C:\Users\Sujeet\AppData\Local\Temp\ipykernel_22812\2888318756.py:2: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders.text import TextLoader


In [7]:
document = loader.load()

In [8]:
document

[Document(metadata={'source': 'data/Python.txt'}, page_content="Python is a high-level, interpreted programming language that has become one of the most popular and widely used languages in the world. Created by Guido van Rossum and first released in 1991, Python emphasizes simplicity and readability, making it easy for beginners to learn while remaining powerful for advanced developers. Its clean and concise syntax allows programmers to write fewer lines of code than in many other languages, enhancing productivity and maintainability. Python supports multiple programming paradigms, including procedural, object-oriented, and functional programming, making it versatile for a wide range of applications.\\nSome key features and benefits of Python include:\\n* Ease of Learning: Simple syntax and readability make Python beginner-friendly.\\n* Versatility: Suitable for web development, data analysis, artificial intelligence, machine learning, scientific computing, automation, and more.\\n* E

## Ingestion Pipeline

In [9]:
# Data => Documents
import os
from langchain_community.document_loaders.pdf import PyPDFLoader

## Documents

In [10]:
def load_all_pdfs():
    folder_path = "data/pdfs"
    num_docs = 0
    all_docs = []

    for filename in os.listdir(folder_path):
        if filename.lower().endswith(".pdf"):
            # complete file path
            pdf_path = os.path.join(folder_path, filename)

            loader = PyPDFLoader(pdf_path)
            doc = loader.load()
            
            all_docs.extend(doc)
            num_docs += 1

    print("total pdfs:", num_docs)
    print("total pages:", len(all_docs))
    return all_docs

In [11]:
all_pdf_documents = load_all_pdfs()

total pdfs: 2
total pages: 32


In [12]:
type(all_pdf_documents[1])

langchain_core.documents.base.Document

## Chunks

In [13]:
# !pip install langchain_text_splitters

In [14]:
from langchain_text_splitters import RecursiveCharacterTextSplitter

def split_docs(documents, chunk_size=500, chunk_overlap=50):
    
    text_splitter = RecursiveCharacterTextSplitter(
        chunk_size = chunk_size,
        chunk_overlap = chunk_overlap
    )

    chunked_docs = text_splitter.split_documents(documents)
    return chunked_docs

In [15]:
chunks = split_docs(all_pdf_documents)

In [16]:
len(chunks)

244

## Embedding

In [17]:
from sentence_transformers import SentenceTransformer

In [18]:
class EmbeddingManager:
    def __init__(self, model_name="all-MiniLM-L6-v2"):
        
        self.model_name=model_name
        print("loading model....", self.model_name)
        self.model = SentenceTransformer(self.model_name)
        print("embedding dimensions=", self.model.get_embedding_dimension())


    def generate_embeddings(self, text):
        embeddings = self.model.encode(text, show_progress_bar=True)
        print("embeddings shape:", embeddings.shape)
        return embeddings

In [19]:
embedding_manager = EmbeddingManager()

loading model.... all-MiniLM-L6-v2


Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

embedding dimensions= 384


## Vector Store 

In [20]:
import chromadb
import uuid

In [21]:
class VectorStoreManager:
    def __init__(self, persist_directory="data/vector_store", collection_name="pdf_documents"):
        self.collection_name = collection_name
        self.persist_directory = persist_directory
        self.collection = None
        self.client = None

        self._initialize_store()

    def _initialize_store(self):
        os.makedirs(self.persist_directory, exist_ok=True)
        
        # create a client
        self.client = chromadb.PersistentClient(path=self.persist_directory)

        # create the collection
        self.collection = self.client.get_or_create_collection(
            name=self.collection_name,
            metadata={"description": "vector store collection for pdf embeddings in RAG"}
        )

        print("initialized the vector store with collection:", self.collection_name)
        print("docs in collection:", self.collection.count())

    def add_documents(self, documents, embeddings):
        if len(documents) != len(embeddings):
            raise ValueError("num of documents does not match num of embeddings")


        # store => ids, embedding, document, metadata
        ids = []
        all_metadata = []
        documents_content = []
        embeddings_list = []

        for i, (doc, embedding) in enumerate(zip(documents, embeddings)):
            doc_id = f"doc_{uuid.uuid4()}"
            ids.append(doc_id)

            metadata = dict(doc.metadata)
            metadata["doc_index"] = i
            metadata["content_length"] = len(doc.page_content)
            all_metadata.append(metadata)

            documents_content.append(doc.page_content)

            embeddings_list.append(embedding.tolist())

            self.collection.add(
                ids=ids,
                metadatas=all_metadata,
                documents=documents_content,
                embeddings=embeddings_list
            )

        print("total documents added in vector store=", len(documents_content))
        print("docs in collection:", self.collection.count())

In [22]:
vector_store = VectorStoreManager()

initialized the vector store with collection: pdf_documents
docs in collection: 488


In [23]:
# data => documents => chunks => embeddings => store in vector store

texts = [doc.page_content for doc in chunks]

emebedding = embedding_manager.generate_embeddings(texts)

vector_store.add_documents(chunks, emebedding)

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

embeddings shape: (244, 384)
total documents added in vector store= 244
docs in collection: 732


# Retrieval Pipeline

In [24]:
from sklearn.metrics.pairwise import cosine_similarity

In [25]:
class RAGRetriever:
    def __init__(self, embedding_manager, vector_store, score_threshold=None):
        self.embedding_manager = embedding_manager
        self.vector_store = vector_store
        self.score_threshold = score_threshold

    def retrieve(self, query, top_k=5, score_threshold=None):
        # Determine effective threshold
        threshold = score_threshold if score_threshold is not None else self.score_threshold

        query_embeddings = self.embedding_manager.generate_embeddings([query])[0]
        
        results = self.vector_store.collection.query(
            query_embeddings=[query_embeddings.tolist()], 
            n_results=top_k
        )
        
        retrieved_docs = []
        if results["documents"] and results["documents"][0]:
            ids = results["ids"][0]
            metadatas = results["metadatas"][0]
            documents = results["documents"][0]
            distances = results["distances"][0]
            
            for i, (doc_id, metadata, document, distance) in enumerate(zip(ids, metadatas, documents, distances)):
                # Convert L2 distance to 0-1 similarity scale
                similarity_score = 1 / (1 + distance)
                
                # Only filter if threshold is explicitly defined
                if threshold is None or similarity_score >= threshold:
                    retrieved_docs.append({
                        "id": doc_id,
                        "document": document,
                        "metadata": metadata,
                        "distance": distance,
                        "similarity_score": similarity_score,
                        "rank": i + 1
                    })
                    
        print(f"retrieved {len(retrieved_docs)} documents")
        return retrieved_docs

In [26]:
rag_retriever = RAGRetriever(embedding_manager, vector_store)

In [27]:
rag_retriever.retrieve("What is encoder decoder")

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

embeddings shape: (1, 384)
retrieved 5 documents


[{'id': 'doc_11aac6c7-416a-4e4d-b98a-7e54e5077f58',
  'document': 'integrity and compression ratio. PRCA tackled this issue by\ntraining an information extractor [69]. Similarly, RECOMP\nadopts a comparable approach by training an information\ncondenser using contrastive learning [71]. Each training data\npoint consists of one positive sample and five negative sam-\nples, and the encoder undergoes training using contrastive loss\nthroughout this process [102] .\nIn addition to compressing the context, reducing the num-',
  'metadata': {'page_label': '10',
   'ptex.fullbanner': 'This is pdfTeX, Version 3.141592653-2.6-1.40.25 (TeX Live 2023) kpathsea version 6.3.5',
   'author': '',
   'producer': 'pdfTeX-1.40.25',
   'source': 'data/pdfs\\ResearchPaper1.pdf',
   'page': 9,
   'keywords': '',
   'creationdate': '2024-03-28T00:54:45+00:00',
   'content_length': 450,
   'total_pages': 21,
   'creator': 'LaTeX with hyperref',
   'moddate': '2024-03-28T00:54:45+00:00',
   'title': '',
   't